# Limpieza de datos
### Dataset: ai_agent_incidents.csv
#### el dataset trata de incidentes y eventos con la AI

# Estructura del notebook
1. Carga de datos y la inspección de ella
2. Limpieza de datos

In [27]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from urllib.parse import urlparse

pd.options.display.max_rows = 10
pd.options.display.max_columns = None

# Carga inicial de datos

In [28]:
datos = pd.read_csv('ai_agent_incidents.csv')
datos.shape

(395, 17)

In [29]:
datos.head()

,incident_id,date,year,month,title,attack_type_code,attack_type_name,incident_kind,severity,confidence_level,real_harm_confirmed,ai_involvement,region,is_landmark_event,summary,source_count,primary_source_url
0,2024-12-01-storm-azure-llmjacking,2024-12-01,2024,2024-12,Storm-2139 Azure OpenAI LLMjacking,CRED,Credential & Secret Theft,incident,high,a,True,confirmed,GLOBAL,False,Microsoft's DCU filed suit in the Eastern Dist...,1,https://blogs.microsoft.com/on-the-issues/2025...
1,2025-01-01-maine-town-deepfake-phishing,2025-01-01,2025,2025-01,AI phishing and deepfake voice hit a small Mai...,OTHER,Unclassified AI Incident,incident,medium,a,True,confirmed,US,False,Impersonating town officials to approve tens o...,1,https://genai.owasp.org/2025/03/06/owasp-gen-a...
2,2025-01-20-hk-voice-clone-usdt-fraud,2025-01-20,2025,2025-01,Hong Kong AI voice-clone crypto fraud,OTHER,Unclassified AI Incident,incident,medium,a,True,confirmed,HK,False,Cloned a finance manager's voice; 3 USDT trans...,1,https://genai.owasp.org/2025/03/06/owasp-gen-a...
3,2025-01-23-operator-fa-bu-bei-jing,2025-01-23,2025,2025-01,OpenAI Operator launches (context entry),GOV,AI Governance & Policy Failure,policy,info,a,False,confirmed,GLOBAL,False,The first consumer-facing computer-use agent; ...,1,https://openai.com/index/operator-system-card/
4,2025-01-26-harbin-asian-winter-games-attack,2025-01-26,2025,2025-01,Harbin Asian Winter Games systems attacked,WEAPON,AI Weaponization & Malicious Tooling,incident,high,d,True,disputed,CN,False,National Computer Virus Emergency Response Cen...,1,http://www.news.cn/sports/20250404/9b6d2457ca4...


In [30]:
datos.dtypes

incident_id             str
date                    str
year                  int64
month                   str
title                   str
                      ...  
region                  str
is_landmark_event      bool
summary                 str
source_count          int64
primary_source_url      str
Length: 17, dtype: object

In [8]:
datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   incident_id          395 non-null    str  
 1   date                 395 non-null    str  
 2   year                 395 non-null    int64
 3   month                395 non-null    str  
 4   title                395 non-null    str  
 5   attack_type_code     395 non-null    str  
 6   attack_type_name     395 non-null    str  
 7   incident_kind        395 non-null    str  
 8   severity             395 non-null    str  
 9   confidence_level     395 non-null    str  
 10  real_harm_confirmed  395 non-null    bool 
 11  ai_involvement       395 non-null    str  
 12  region               395 non-null    str  
 13  is_landmark_event    395 non-null    bool 
 14  summary              395 non-null    str  
 15  source_count         395 non-null    int64
 16  primary_source_url   395 non-null    

# Valores faltantes, duplicados

In [31]:
print("Valores faltantes por columna:")
print(datos.isna().sum())
print("\nFilas duplicadas:", datos.duplicated().sum())
print("incident_id duplicados:", datos['incident_id'].duplicated().sum())

# Cadenas vacías o solo con espacios
cols_texto = datos.select_dtypes(include=['object', 'string']).columns
print("\nCadenas vacías:", (datos[cols_texto].apply(lambda s: s.str.strip() == '')).sum().sum())


Valores faltantes por columna:
incident_id           0
date                  0
year                  0
month                 0
title                 0
                     ..
region                0
is_landmark_event     0
summary               0
source_count          0
primary_source_url    0
Length: 17, dtype: int64

Filas duplicadas: 0
incident_id duplicados: 0

Cadenas vacías: 0


In [32]:
datos.nunique()

incident_id           395
date                  241
year                    3
month                  23
title                 395
                     ... 
region                 19
is_landmark_event       2
summary               395
source_count            1
primary_source_url    353
Length: 17, dtype: int64

# limpieza de datos

In [33]:
### aqui vemos espacios en blanco en columnas de textos
for c in cols_texto:
    datos[c] = datos[c].str.strip()

In [34]:
## aqui se hace el cambio de las fechas
datos['date'] = pd.to_datetime(datos['date'], errors='coerce')

print("Fechas inválidas:", datos['date'].isna().sum())
print("year distinto al año de date:", (datos['date'].dt.year != datos['year']).sum())
print("month distinto al mes de date:", (datos['date'].dt.strftime('%Y-%m') != datos['month']).sum())
print("Rango de fechas:", datos['date'].min().date(), "->", datos['date'].max().date())

Fechas inválidas: 0
year distinto al año de date: 0
month distinto al mes de date: 0
Rango de fechas: 2024-12-01 -> 2026-10-09


In [35]:
### aqui esta la columna que es muy constante y su valor es 1 en todas las filas
print(datos['source_count'].value_counts())
datos = datos.drop(columns='source_count')

source_count
1    395
Name: count, dtype: int64


In [36]:
##Columna redundante: `attack_type_name` vs `attack_type_code`
# Verificamos que la relación código <-> nombre sea uno a uno
print(datos.groupby('attack_type_code')['attack_type_name'].nunique().max())
print(datos.groupby('attack_type_name')['attack_type_code'].nunique().max())

# Guardamos un diccionario código -> nombre para poder rotular gráficos si hace falta y quitamos la columna
dic_tipos = datos.drop_duplicates('attack_type_code').set_index('attack_type_code')['attack_type_name'].to_dict()
datos = datos.drop(columns='attack_type_name')

1
1


## Columnas multietiqueta: attack_type_code y region
Varias filas traen más de un valor (por ejemplo IPI;EXFIL o CN;GLOBAL). Se crean variables para contar cuántos valores tiene cada fila y se conserva la columna original para poder "explotarla" en el EDA.


In [37]:
#### 2.6 Texto y fuentes: `summary` y `primary_source_url`

# Quitamos las marcas de Markdown (**negrita**) y normalizamos espacios
datos['resumen_limpio'] = (datos['summary']
                           .str.replace('**', '', regex=False)
                           .str.replace(r'\s+', ' ', regex=True)
                           .str.strip())
datos['longitud_resumen'] = datos['resumen_limpio'].str.len()
datos = datos.drop(columns='summary')

# URLs: la columna puede traer varias separadas por ';'
datos['n_fuentes'] = datos['primary_source_url'].str.count(';') + 1
datos['url_principal'] = datos['primary_source_url'].str.split(';').str[0]
datos['dominio'] = datos['url_principal'].apply(lambda u: urlparse(u).netloc.replace('www.', ''))

datos[['resumen_limpio', 'longitud_resumen', 'n_fuentes', 'dominio']].head()

,resumen_limpio,longitud_resumen,n_fuentes,dominio
0,Microsoft's DCU filed suit in the Eastern Dist...,281,1,blogs.microsoft.com
1,Impersonating town officials to approve tens o...,80,1,genai.owasp.org
2,Cloned a finance manager's voice; 3 USDT trans...,92,1,genai.owasp.org
3,The first consumer-facing computer-use agent; ...,351,1,openai.com
4,National Computer Virus Emergency Response Cen...,548,3,news.cn


In [38]:
# Tipos de datos: variables ordinales
#`severity` y `confidence_level` tienen un orden natural, así que se convierten a categóricas ordenadas.
orden_sev = ['info', 'low', 'medium', 'high', 'critical']
datos['severity'] = pd.Categorical(datos['severity'], categories=orden_sev, ordered=True)
datos['confidence_level'] = pd.Categorical(datos['confidence_level'], categories=['a', 'b', 'c', 'd'], ordered=True)

# Texto -> categoría para las demás variables cualitativas
for c in ['incident_kind', 'ai_involvement']:
    datos[c] = datos[c].astype('category')

datos.dtypes

incident_id                    str
date                datetime64[us]
year                         int64
month                          str
title                          str
                         ...      
resumen_limpio                 str
longitud_resumen             int64
n_fuentes                    int64
url_principal               object
dominio                        str
Length: 19, dtype: object

In [39]:
#Las filas con severidad `info` son entradas de contexto (`policy` y `report`), no incidentes técnicos. Se marcan con una variable para poder analizarlas por separado.
datos['es_contexto'] = datos['severity'] == 'info'
print(pd.crosstab(datos['incident_kind'], datos['es_contexto']))

eventos = datos[~datos['es_contexto']].copy()   # solo eventos con severidad real
eventos.shape

es_contexto    False  True 
incident_kind              
incident         162      0
policy             0     72
report             0     15
research          91      0
vulnerability     55      0


(308, 20)

In [24]:
q1, q3 = datos['longitud_resumen'].quantile([.25, .75])
iqr = q3 - q1
limite_sup = q3 + 1.5 * iqr
atipicos = datos[datos['longitud_resumen'] > limite_sup]
print("Límite superior:", limite_sup, "| atípicos:", len(atipicos))

Límite superior: 1464.25 | atípicos: 11


In [40]:
# este es el resumen de la limpieza
print(datos.shape)
print(datos.isna().sum().sum(), "valores faltantes")
datos.to_csv('ai_agent_incidents_limpio.csv', index=False)
datos.head()

(395, 20)
0 valores faltantes


,incident_id,date,year,month,title,attack_type_code,incident_kind,severity,confidence_level,real_harm_confirmed,ai_involvement,region,is_landmark_event,primary_source_url,resumen_limpio,longitud_resumen,n_fuentes,url_principal,dominio,es_contexto
0,2024-12-01-storm-azure-llmjacking,2024-12-01,2024,2024-12,Storm-2139 Azure OpenAI LLMjacking,CRED,incident,high,a,True,confirmed,GLOBAL,False,https://blogs.microsoft.com/on-the-issues/2025...,Microsoft's DCU filed suit in the Eastern Dist...,281,1,https://blogs.microsoft.com/on-the-issues/2025...,blogs.microsoft.com,False
1,2025-01-01-maine-town-deepfake-phishing,2025-01-01,2025,2025-01,AI phishing and deepfake voice hit a small Mai...,OTHER,incident,medium,a,True,confirmed,US,False,https://genai.owasp.org/2025/03/06/owasp-gen-a...,Impersonating town officials to approve tens o...,80,1,https://genai.owasp.org/2025/03/06/owasp-gen-a...,genai.owasp.org,False
2,2025-01-20-hk-voice-clone-usdt-fraud,2025-01-20,2025,2025-01,Hong Kong AI voice-clone crypto fraud,OTHER,incident,medium,a,True,confirmed,HK,False,https://genai.owasp.org/2025/03/06/owasp-gen-a...,Cloned a finance manager's voice; 3 USDT trans...,92,1,https://genai.owasp.org/2025/03/06/owasp-gen-a...,genai.owasp.org,False
3,2025-01-23-operator-fa-bu-bei-jing,2025-01-23,2025,2025-01,OpenAI Operator launches (context entry),GOV,policy,info,a,False,confirmed,GLOBAL,False,https://openai.com/index/operator-system-card/,The first consumer-facing computer-use agent; ...,351,1,https://openai.com/index/operator-system-card/,openai.com,True
4,2025-01-26-harbin-asian-winter-games-attack,2025-01-26,2025,2025-01,Harbin Asian Winter Games systems attacked,WEAPON,incident,high,d,True,disputed,CN,False,http://www.news.cn/sports/20250404/9b6d2457ca4...,National Computer Virus Emergency Response Cen...,548,3,http://www.news.cn/sports/20250404/9b6d2457ca4...,news.cn,False
